In [19]:
# build_index.py

import os
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import numpy as np

In [20]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
pdf_folders = [
    "data/Healthy",
    "data/Schizophrenia"
]

documents = []

for folder in pdf_folders:

    label = os.path.basename(folder)

    for file in os.listdir(folder):

        if file.endswith(".pdf"):

            path = os.path.join(folder, file)

            pdf = fitz.open(path)

            text = ""

            for page in pdf:
                text += page.get_text()

            documents.append({
                "filename": file,
                "label": label,
                "text": text
            })

In [22]:
print(len(documents))

for d in documents[:50]:
    print(d["filename"], d["label"])

92
C001.pdf Healthy
C002.pdf Healthy
C003 .pdf Healthy
C004.pdf Healthy
C005.pdf Healthy
C006.pdf Healthy
C007.pdf Healthy
C008.rnfl.pdf Healthy
C009.pdf Healthy
C010.pdf Healthy
C011.pdf Healthy
C012.rnfl.pdf Healthy
C013.rnfl.pdf Healthy
C014.rnfl.pdf Healthy
C015.rnfl.pdf Healthy
C016.rnfl .pdf Healthy
C017.rnfl.pdf Healthy
C018.rnfl.pdf Healthy
C019.rnfl.pdf Healthy
C020rnfl.pdf Healthy
C021.rnfl.pdf Healthy
C022.rnfl.pdf Healthy
C023.rnfl.pdf Healthy
C024.rnfl.pdf Healthy
C025rnfl.pdf Healthy
C026.rnfl.pdf Healthy
C027.rnfl.pdf Healthy
C028.rnfl.pdf Healthy
C029.rnfl.pdf Healthy
C030.rnfl.pdf Healthy
C031.rnfl.pdf Healthy
C032.rnfl.pdf Healthy
C033.rnfl.pdf Healthy
C034.rnfl.pdf Healthy
C035.rnfl.pdf Healthy
C036.rnfl.pdf Healthy
C037.rnfl.pdf Healthy
C038.rnfl.pdf Healthy
C039.rnfl.pdf Healthy
C040.rnfl.pdf Healthy
C041.rnfl.pdf Healthy
C042.rnfl.pdf Healthy
C043.rnfl.pdf Healthy
C044.rnfl.pdf Healthy
C045.rnfl.pdf Healthy
C046.rnfl.pdf Healthy
C047.rnfl.pdf Healthy
IAS001.pdf Sc

In [23]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = []
metadata = []

for doc in documents:

    pieces = splitter.split_text(doc["text"])

    for p in pieces:

        chunk_text = f"""
GROUP: {doc['label']}

FILE: {doc['filename']}

{p}
"""

        chunks.append(chunk_text)

        metadata.append({
            "filename": doc["filename"],
            "label": doc["label"]
        })

In [24]:
print(len(chunks))

406


In [25]:
embeddings = embedder.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [26]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

In [27]:
os.makedirs("vectorstore", exist_ok=True)

faiss.write_index(
    index,
    "vectorstore/oct_index.faiss"
)

with open(
    "vectorstore/metadata.pkl",
    "wb"
) as f:

    pickle.dump(
        {
            "chunks": chunks,
            "metadata": metadata
        },
        f
    )